# Nhiệm vụ 6: Thực thi PCA + K-Means

## Mục đích
- Giảm chiều dữ liệu từ 8 features xuống số lượng principal components tối thiểu sao cho cumulative explained variance >= 90%.
- Freeze số lượng component này (kết quả là 4 components) cho toàn bộ tập development.
- Thực thi K-Means trên không gian PCA với Global K.
- Lưu lại toàn bộ mô hình (PCA transformers, KMeans models) và artifacts (assignments, profiles, diagnostics).


In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# Đảm bảo working directory ở root của dự án
while not os.getcwd().endswith('m2-clustering-experiment'):
    os.chdir('..')

print("Current Working Directory:", os.getcwd())


## 1. Cấu hình Experiment PCA + K-Means

In [ ]:
config_path = 'configs/experiments/m2_task6_pca.json'

with open(config_path, 'r', encoding='utf-8') as f:
    config = json.load(f)

print("Algorithm:", config['clustering']['algorithm'])
print("Features:", config['clustering']['features'])
print("PCA Config:", config['clustering']['pca'])


## 2. Khởi chạy Pipeline (Thực thi Runner)

In [ ]:
from src.delta_t1.experiments.runner import experiment

# Chạy runner với cấu hình của Nhiệm vụ 6
# Output sẽ được lưu tự động vào M2/artifacts/m2-task6-pca-kmeans-v1/
data_run = Path('artifacts/cafef_primary/cafef-c8-complete-only-v1')
config_p = Path(config_path)
root_p = Path('.')

target_dir, manifest = experiment(data_run, config_p, root_p)
print("Pipeline executed successfully!")
print("Target Output:", target_dir)


## 3. Kiểm tra Kết quả (Explained Variance & Profiles)

In [ ]:
# Xem thử một trong các mô hình PCA đã lưu
import joblib
pca_model_path = 'M2/models/pca_kmeans/2023-11-30.json'

with open(pca_model_path, 'r', encoding='utf-8') as f:
    model_data = json.load(f)

print("PCA Components Freeze:", model_data.get('n_components', 4))
if 'pca_explained_variance_ratio' in model_data:
    ratios = model_data['pca_explained_variance_ratio']
    print("Explained Variance Ratios:", ratios)
    print("Cumulative Explained Variance:", sum(ratios))


In [ ]:
# Kiểm tra Artifacts được tạo ra
artifacts_dir = 'M2/artifacts/m2-task6-pca-kmeans-v1'
assignments_path = f'{artifacts_dir}/assignments.csv'
profiles_path = f'{artifacts_dir}/profiles.csv'

if os.path.exists(profiles_path):
    df_profiles = pd.read_csv(profiles_path)
    display(df_profiles.head())
else:
    print("Artifacts chưa được tạo!")
